# Generazione di posizioni di scacchi — ChessVQVAE

Questo notebook esplora la **generazione** nello spazio latente del VQ-VAE.

**Tre modalità di generazione:**
1. **Random sampling dal codebook** — campiona combinazioni casuali di vettori
2. **Guided sampling** — campiona condizionato su valutazione target (se il modello ha la loss ausiliaria)
3. **Interpolazione** — muoviti tra due posizioni reali nello spazio latente
4. **Neighbourhood** — esplora le posizioni vicine a una posizione reale

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import chess
import chess.svg
from IPython.display import display, SVG, HTML
import warnings
warnings.filterwarnings('ignore')

import sys, os
current_dir = os.getcwd()
if current_dir in sys.path:
    sys.path.remove(current_dir)
    sys.path.append(current_dir)

from vqvae_chess import ChessVQVAE
from MLChess import ChessTransform, generate_all_legal_move_vocab

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# ── CONFIGURA QUI ──────────────────────────────────────────
CHECKPOINT_PATH = './runs/vqvae/checkpoints/best.pt'
# ───────────────────────────────────────────────────────────

ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
cfg  = ckpt['cfg']

model = ChessVQVAE(
    latent_dim     = cfg['latent_dim'],
    num_embeddings = cfg['num_embeddings'],
    base_ch        = cfg['base_ch'],
    beta           = cfg['beta'],
    focal_alpha    = cfg['focal_alpha'],
    focal_gamma    = cfg['focal_gamma'],
    aux_weight     = cfg.get('aux_weight', 0.0),
).to(device)
model.load_state_dict(ckpt['model'])
model.eval()

K = cfg['num_embeddings']
D = cfg['latent_dim']
print(f'Modello caricato — K={K}, D={D}, epoch={ckpt["epoch"]}')

In [ ]:
# Utilità condivise

PIECE_MAP = {
    0:  chess.Piece(chess.PAWN,   chess.WHITE),
    1:  chess.Piece(chess.KNIGHT, chess.WHITE),
    2:  chess.Piece(chess.BISHOP, chess.WHITE),
    3:  chess.Piece(chess.ROOK,   chess.WHITE),
    4:  chess.Piece(chess.QUEEN,  chess.WHITE),
    5:  chess.Piece(chess.KING,   chess.WHITE),
    6:  chess.Piece(chess.PAWN,   chess.BLACK),
    7:  chess.Piece(chess.KNIGHT, chess.BLACK),
    8:  chess.Piece(chess.BISHOP, chess.BLACK),
    9:  chess.Piece(chess.ROOK,   chess.BLACK),
    10: chess.Piece(chess.QUEEN,  chess.BLACK),
    11: chess.Piece(chess.KING,   chess.BLACK),
}

def tensor_to_board(t: torch.Tensor) -> chess.Board:
    """Converte un tensore ricostruito (14,8,8) in una chess.Board.
    
    Usa argmax sui 13 logit categorici per casella (classe 0=vuoto, 1-12=pezzi).
    Garantisce strutturalmente al massimo un pezzo per casella.
    """
    board = chess.Board(None)
    # t[:13] = logit categorici (B,13,8,8) oppure (13,8,8)
    logits = t[:13].cpu()
    # argmax sul canale: 0=vuoto, 1-12=pezzo
    piece_idx = logits.argmax(dim=0).numpy()  # (8, 8), valori 0-12
    for rank in range(8):
        for file in range(8):
            sq  = chess.square(file, 7 - rank)
            cls = int(piece_idx[rank, file])
            if cls > 0:  # 0 = vuoto
                board.set_piece_at(sq, PIECE_MAP[cls - 1])
    return board


def decode_indices(indices: torch.Tensor) -> torch.Tensor:
    """Decodifica indici VQ (B, 4) → posizioni (B, 13, 8, 8) sigmoid."""
    B = indices.shape[0]
    with torch.no_grad():
        # Lookup nel codebook
        emb = model.vq.embedding.weight  # (K, D) normalizzato
        z_q_flat = emb[indices.reshape(-1)]          # (B*4, D)
        z_q_flat = model.vq.post_proj(z_q_flat)      # (B*4, D)
        z_q = z_q_flat.reshape(B, 2, 2, D).permute(0, 3, 1, 2)  # (B, D, 2, 2)
        x_recon = torch.sigmoid(model.decoder(z_q))  # (B, 13, 8, 8)
    return x_recon


def show_boards(boards, titles=None, cols=4, size=200):
    """Mostra una griglia di scacchiere SVG."""
    n = len(boards)
    rows = (n + cols - 1) // cols
    html = '<div style="display:flex; flex-wrap:wrap; gap:10px;">'
    for i, board in enumerate(boards):
        title = titles[i] if titles else f'#{i}'
        svg   = chess.svg.board(board, size=size)
        html += f'<div style="text-align:center"><div style="font-size:11px;color:#aaa;margin-bottom:3px">{title}</div>{svg}</div>'
    html += '</div>'
    display(HTML(html))


def board_stats(board: chess.Board) -> str:
    """Stringa di statistiche rapide per una scacchiera."""
    pieces = board.piece_map()
    n_w = sum(1 for p in pieces.values() if p.color == chess.WHITE)
    n_b = sum(1 for p in pieces.values() if p.color == chess.BLACK)
    return f'W:{n_w} B:{n_b} tot:{len(pieces)}'


# Distribuzione empirica degli indici (per il sampling pesato)
# Stima la prior P(indice) dal training set in memoria
codebook_weights = model.vq.ema_cluster_size.detach().cpu()
codebook_weights = (codebook_weights / codebook_weights.sum()).numpy()
print('Utilità caricate')
print(f'Vettori con peso > 0: {(codebook_weights > 1e-6).sum()} / {K}')

## 1. Random sampling dal codebook

Ogni posizione è codificata da **4 indici** (spazio latente 2×2).
Campioniamo 4 indici casuali e decodifichiamo.

In [ ]:
def generate_random(n: int = 8, temperature: float = 1.0,
                    use_prior: bool = True) -> list:
    """
    Genera n posizioni campionando indici casuali dal codebook.
    
    temperature: > 1 = più uniforme (più diversità), < 1 = più concentrato
    use_prior:   True = campiona proporzionalmente all'utilizzo nel training
                 False = uniforme su tutti i K vettori
    """
    if use_prior:
        w = codebook_weights ** (1.0 / temperature)
        w = w / w.sum()
        indices = np.random.choice(K, size=(n, 4), p=w)
    else:
        indices = np.random.randint(0, K, size=(n, 4))
    
    idx_tensor = torch.tensor(indices, dtype=torch.long).to(device)
    recons     = decode_indices(idx_tensor)   # (n, 13, 8, 8)
    return [tensor_to_board(recons[i]) for i in range(n)]


print('── Sampling uniforme (temperature=1, prior=True) ──')
boards_t1 = generate_random(8, temperature=1.0, use_prior=True)
show_boards(boards_t1, [board_stats(b) for b in boards_t1])

In [ ]:
print('── Sampling "freddo" (temperature=0.5 — più simile al training) ──')
boards_cold = generate_random(8, temperature=0.5, use_prior=True)
show_boards(boards_cold, [board_stats(b) for b in boards_cold])

print('── Sampling "caldo" (temperature=2.0 — più diversità) ──')
boards_hot = generate_random(8, temperature=2.0, use_prior=True)
show_boards(boards_hot, [board_stats(b) for b in boards_hot])

## 2. Analisi della "realismo" delle posizioni generate

Una posizione generata può essere più o meno realistica.
Calcoliamo alcune metriche di validità.

In [ ]:
def board_validity_score(board: chess.Board) -> dict:
    """Valuta quanto una posizione generata è 'realistica'."""
    pieces = board.piece_map()
    
    # Conta per tipo
    counts = {pt: {c: 0 for c in [chess.WHITE, chess.BLACK]}
              for pt in chess.PIECE_TYPES}
    for p in pieces.values():
        counts[p.piece_type][p.color] += 1
    
    issues = []
    score  = 1.0
    
    for color in [chess.WHITE, chess.BLACK]:
        name = 'Bianco' if color == chess.WHITE else 'Nero'
        
        # Re: deve essere esattamente 1
        if counts[chess.KING][color] != 1:
            issues.append(f'{name}: {counts[chess.KING][color]} re')
            score -= 0.4
        
        # Pedoni: max 8
        if counts[chess.PAWN][color] > 8:
            issues.append(f'{name}: {counts[chess.PAWN][color]} pedoni')
            score -= 0.2
        
        # Regine: max 9 (1 originale + 8 promozioni), ma > 2 è strano
        if counts[chess.QUEEN][color] > 2:
            issues.append(f'{name}: {counts[chess.QUEEN][color]} regine')
            score -= 0.1
        
        # Totale pezzi per colore: max 16
        total_color = sum(counts[pt][color] for pt in chess.PIECE_TYPES)
        if total_color > 16:
            issues.append(f'{name}: {total_color} pezzi totali')
            score -= 0.3
    
    # Posizioni vuote/sparse: meno di 4 pezzi totali è strano
    if len(pieces) < 4:
        issues.append(f'Troppo pochi pezzi: {len(pieces)}')
        score -= 0.2
    
    return {
        'score':  max(0.0, score),
        'issues': issues,
        'n_pieces': len(pieces),
        'valid': len(issues) == 0,
    }


# Genera 200 posizioni e analizza il realismo
N_GEN = 200
boards_analysis = generate_random(N_GEN, temperature=1.0, use_prior=True)
scores = [board_validity_score(b) for b in boards_analysis]

n_valid      = sum(1 for s in scores if s['valid'])
mean_score   = np.mean([s['score'] for s in scores])
mean_pieces  = np.mean([s['n_pieces'] for s in scores])

print(f'Posizioni generate:          {N_GEN}')
print(f'Posizioni "valide":           {n_valid} ({100*n_valid/N_GEN:.0f}%)')
print(f'Score medio di realismo:      {mean_score:.3f} / 1.0')
print(f'Numero medio pezzi:           {mean_pieces:.1f}')

# Problemi più comuni
all_issues = [issue for s in scores for issue in s['issues']]
from collections import Counter
issue_counts = Counter(all_issues)
print('\nProblemi più comuni:')
for issue, cnt in issue_counts.most_common(10):
    print(f'  {cnt:4d}x  {issue}')

# Distribuzione numero pezzi
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
piece_counts_gen = [s['n_pieces'] for s in scores]
axes[0].hist(piece_counts_gen, bins=20, alpha=0.7, color='steelblue')
axes[0].axvline(mean_pieces, color='red', linestyle='--', label=f'Media: {mean_pieces:.1f}')
axes[0].set_title('Distribuzione numero pezzi — posizioni generate')
axes[0].set_xlabel('Numero pezzi')
axes[0].legend()

score_vals = [s['score'] for s in scores]
axes[1].hist(score_vals, bins=20, alpha=0.7, color='orange')
axes[1].axvline(mean_score, color='red', linestyle='--', label=f'Media: {mean_score:.3f}')
axes[1].set_title('Distribuzione score realismo')
axes[1].set_xlabel('Score (1.0 = perfetto)')
axes[1].legend()

plt.tight_layout()
plt.show()

# Mostra le 6 posizioni con score più alto
best_idx = np.argsort([-s['score'] for s in scores])[:6]
print('\n── Le 6 posizioni generate più realistiche ──')
show_boards(
    [boards_analysis[i] for i in best_idx],
    [f"score={scores[i]['score']:.2f} pezzi={scores[i]['n_pieces']}" for i in best_idx]
)

## 3. Interpolazione tra due posizioni reali

In [ ]:
all_moves = generate_all_legal_move_vocab()
transform = ChessTransform(move_vocab=all_moves)

def fen_to_tensor(fen: str) -> torch.Tensor:
    board = chess.Board(fen)
    mv = list(board.legal_moves)[0].uci() if board.legal_moves else 'e2e4'
    t, _, _, _ = transform(fen, mv, [], '0')
    return t.unsqueeze(0).to(device)


def interpolate(fen_a: str, fen_b: str, n_steps: int = 9,
                threshold: float = 0.35):
    """
    Interpola nello spazio z_e (pre-VQ) tra due posizioni.
    Ogni punto viene poi quantizzato e decodificato.
    """
    with torch.no_grad():
        za = model.encoder(fen_to_tensor(fen_a))  # (1, D, 2, 2)
        zb = model.encoder(fen_to_tensor(fen_b))
    
    boards_interp = []
    titles        = []
    for i, t in enumerate(np.linspace(0, 1, n_steps)):
        z_interp = ((1 - t) * za + t * zb).float()
        with torch.no_grad():
            z_q, _, _, _ = model.vq(z_interp)
            recon = torch.sigmoid(model.decoder(z_q))[0]  # (13, 8, 8)
        boards_interp.append(tensor_to_board(recon, threshold))
        titles.append(f't={t:.2f}')
    
    return boards_interp, titles


# Esempio posizioni — modifica a piacere
FEN_START  = 'rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1'  # posizione iniziale
FEN_LATE   = '8/8/4k3/8/8/4K3/8/8 w - - 0 1'                              # finale re vs re

print('Interpolazione: posizione iniziale → finale re vs re')
boards_i, titles_i = interpolate(FEN_START, FEN_LATE, n_steps=9)
show_boards(boards_i, titles_i, cols=9, size=140)

print('\nOriginale A:')
display(SVG(chess.svg.board(chess.Board(FEN_START), size=200)))
print('Originale B:')
display(SVG(chess.svg.board(chess.Board(FEN_LATE), size=200)))

In [ ]:
# Interpolazione tra due posizioni tattiche reali — carica dal CSV
import pandas as pd

CSV_PATH = '../over_mate_1_tactic_evals.csv'  # modifica se necessario
df = pd.read_csv(CSV_PATH).sample(1000, random_state=42)

# Prendi una posizione con vantaggio bianco e una con vantaggio nero
white_fens = df[df['Evaluation'].astype(str).str.replace('+','',regex=False).apply(
    lambda x: float(x) > 0.5 if not '#' in x else True
)]['FEN'].values
black_fens = df[df['Evaluation'].astype(str).str.replace('+','',regex=False).apply(
    lambda x: float(x) < -0.5 if not '#' in x else False
)]['FEN'].values

if len(white_fens) > 0 and len(black_fens) > 0:
    fen_w = white_fens[0]
    fen_b = black_fens[0]
    print(f'Posizione bianco: {fen_w}')
    print(f'Posizione nero:   {fen_b}')
    print('\nInterpolazione: vantaggio bianco → vantaggio nero')
    boards_wb, titles_wb = interpolate(fen_w, fen_b, n_steps=9)
    show_boards(boards_wb, titles_wb, cols=9, size=140)
    print('\nOriginale A (vantaggio bianco):')
    display(SVG(chess.svg.board(chess.Board(fen_w), size=220)))
    print('Originale B (vantaggio nero):')
    display(SVG(chess.svg.board(chess.Board(fen_b), size=220)))

## 4. Neighbourhood — variazioni locali di una posizione

In [ ]:
def generate_neighbourhood(fen: str, n: int = 8,
                            noise_std: float = 0.3,
                            threshold: float = 0.35):
    """
    Genera variazioni di una posizione aggiungendo rumore gaussiano
    allo spazio latente z_e prima della quantizzazione.
    
    noise_std piccolo  → variazioni simili all'originale
    noise_std grande   → variazioni più diverse
    """
    with torch.no_grad():
        z = model.encoder(fen_to_tensor(fen))  # (1, D, 2, 2)
    
    boards   = []
    for _ in range(n):
        noise   = torch.randn_like(z) * noise_std
        z_noisy = (z + noise).float()
        with torch.no_grad():
            z_q, _, _, _ = model.vq(z_noisy)
            recon = torch.sigmoid(model.decoder(z_q))[0]
        boards.append(tensor_to_board(recon, threshold))
    return boards


# Posizione di partenza
FEN_TEST = 'r1bqkb1r/pppp1ppp/2n2n2/4p3/2B1P3/5N2/PPPP1PPP/RNBQK2R w KQkq - 4 4'  # Giuoco Piano

print('Posizione originale:')
display(SVG(chess.svg.board(chess.Board(FEN_TEST), size=220)))

print('\n── Variazioni con rumore piccolo (std=0.2) ──')
boards_small = generate_neighbourhood(FEN_TEST, n=8, noise_std=0.2)
show_boards(boards_small, [board_stats(b) for b in boards_small])

print('\n── Variazioni con rumore medio (std=0.5) ──')
boards_med = generate_neighbourhood(FEN_TEST, n=8, noise_std=0.5)
show_boards(boards_med, [board_stats(b) for b in boards_med])

print('\n── Variazioni con rumore grande (std=1.0) ──')
boards_large = generate_neighbourhood(FEN_TEST, n=8, noise_std=1.0)
show_boards(boards_large, [board_stats(b) for b in boards_large])

## 5. Guided sampling (solo se il modello ha la loss ausiliaria)

Se hai addestrato con `--aux_weight > 0`, lo spazio latente ha una
direzione semantica legata alla valutazione. Possiamo campionare
condizionato su un target di valutazione.

In [ ]:
HAS_AUX = cfg.get('aux_weight', 0.0) > 0 and hasattr(model, 'eval_head')
print(f'Loss ausiliaria disponibile: {HAS_AUX}')

if HAS_AUX:
    def predict_eval(fen: str) -> float:
        """Predice la valutazione di una posizione tramite la testa ausiliaria."""
        with torch.no_grad():
            z_e = model.encoder(fen_to_tensor(fen))
            z_q, _, _, _ = model.vq(z_e.float())
            z_q_flat = z_q.reshape(1, -1).float()
            pred = model.eval_head(z_q_flat).item()
        return pred
    
    # Testa la predizione su qualche FEN noto
    test_fens = [
        ('rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1', 0.0, 'Inizio partita'),
        ('8/8/4k3/8/8/4K3/8/8 w - - 0 1', 0.0, 'Re vs Re'),
        ('QQQQQQQQ/QQQQQQQQ/8/8/8/8/8/4k2K w - - 0 1', 1.0, 'Massimo vantaggio bianco (artificiale)'),
    ]
    print('Predizioni valutazione:')
    for fen, expected, desc in test_fens:
        try:
            pred = predict_eval(fen)
            print(f'  {desc}: pred={pred:+.3f} (atteso≈{expected:+.1f})')
        except:
            print(f'  {desc}: FEN non valida, skip')
else:
    print('\nPer abilitare il guided sampling, riaddestra con --aux_weight 0.5')
    print('Lo spazio latente avrà allora una struttura semantica esplicita.')

In [ ]:
if HAS_AUX:
    # Gradient-guided sampling: ottimizza gli indici per avvicinarsi
    # a un target di valutazione tramite ottimizzazione nello spazio continuo
    def guided_generate(eval_target: float, n_steps: int = 50,
                         lr: float = 0.05):
        """
        Genera una posizione con valutazione target ottimizzando
        z_e nello spazio continuo, poi quantizzando.
        
        eval_target: in [-1, 1], positivo = vantaggio bianco
        """
        # Inizia da un z_e casuale
        z = torch.randn(1, D, 2, 2, device=device) * 0.1
        z.requires_grad_(True)
        opt = torch.optim.Adam([z], lr=lr)
        target = torch.tensor([[eval_target]], dtype=torch.float32, device=device)
        
        for step in range(n_steps):
            opt.zero_grad()
            z_q, _, commit, _ = model.vq(z.float())
            z_q_flat = z_q.reshape(1, -1).float()
            pred = model.eval_head(z_q_flat)
            loss = (pred - target).pow(2) + 0.1 * commit
            loss.backward()
            opt.step()
        
        with torch.no_grad():
            z_q_final, _, _, _ = model.vq(z.float())
            recon = torch.sigmoid(model.decoder(z_q_final))[0]
            pred_final = model.eval_head(z_q_final.reshape(1, -1).float()).item()
        
        board = tensor_to_board(recon, threshold)
        return board, pred_final
    
    targets = [-0.8, -0.4, 0.0, 0.4, 0.8]
    results = []
    preds   = []
    for t in targets:
        board, pred = guided_generate(t, n_steps=100)
        results.append(board)
        preds.append(pred)
        print(f'Target: {t:+.1f}  →  Predetto: {pred:+.3f}  Pezzi: {len(board.piece_map())}')
    
    show_boards(results,
                [f'target={t:+.1f}\npred={p:+.3f}' for t, p in zip(targets, preds)])

## 6. Statistiche aggregate sulla generazione

In [ ]:
# Confronto tra posizioni reali e generate sulle stesse metriche
import pandas as pd

CSV_PATH = '../over_mate_1_tactic_evals.csv'
df_real = pd.read_csv(CSV_PATH).sample(500, random_state=0)

def pieces_from_fen(fen):
    try:
        return len(chess.Board(fen).piece_map())
    except:
        return 0

real_piece_counts = df_real['FEN'].apply(pieces_from_fen).values

# Genera 500 posizioni
gen_boards = generate_random(500, temperature=1.0, use_prior=True)
gen_piece_counts = np.array([len(b.piece_map()) for b in gen_boards])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(real_piece_counts, bins=20, alpha=0.6, label='Reali', color='steelblue')
axes[0].hist(gen_piece_counts,  bins=20, alpha=0.6, label='Generate', color='orange')
axes[0].set_title('Distribuzione numero pezzi: reali vs generate')
axes[0].set_xlabel('Numero pezzi')
axes[0].legend()
axes[0].axvline(real_piece_counts.mean(), color='blue',  linestyle='--', alpha=0.8,
                label=f'Media reali: {real_piece_counts.mean():.1f}')
axes[0].axvline(gen_piece_counts.mean(),  color='red',   linestyle='--', alpha=0.8,
                label=f'Media generate: {gen_piece_counts.mean():.1f}')

gen_scores = [board_validity_score(b)['score'] for b in gen_boards]
axes[1].hist(gen_scores, bins=20, color='green', alpha=0.7)
axes[1].axvline(np.mean(gen_scores), color='red', linestyle='--',
                label=f'Media: {np.mean(gen_scores):.3f}')
axes[1].set_title('Score validità delle posizioni generate')
axes[1].set_xlabel('Score (0=invalida, 1=perfetta)')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'\nPezzi medi reali:      {real_piece_counts.mean():.1f} ± {real_piece_counts.std():.1f}')
print(f'Pezzi medi generate:   {gen_piece_counts.mean():.1f} ± {gen_piece_counts.std():.1f}')
print(f'Score validità medio:  {np.mean(gen_scores):.3f}')
print(f'Posizioni valide:      {sum(1 for s in gen_scores if s == 1.0)} / 500')